# 01 — Diagnosis Kualitas Data
### Milestone M2 · Exploratory Data Analysis (KMTI21133)

| | |
|---|---|
| **Nama tim** | DANVA |
| **Anggota** | Diyanatul Fauziyah, Nasya Awaliyah, Raveska Hinayah, Asysyifa Ainaya Zahra |
| **Topik proyek** | Analisis Pola Penjualan Supermarket Berdasarkan Produk, Kota, Pelanggan, dan Metode Pembayaran |
| **Tanggal** | 15 November 2026 |

**Notebook ini mengisi:** dimensi **Kelengkapan diagnosis kualitas data** (bobot 25) pada Rubrik M2.

Tujuan notebook ini **hanya mendiagnosis** — belum memperbaiki apa pun. Pembersihan
dikerjakan di `02_cleaning.ipynb`. Memisahkan keduanya penting supaya temuan dapat
diperiksa ulang tanpa tercampur dengan perubahan yang sudah terlanjur dilakukan.

Periksa **kelima** dimensi di bawah. Memeriksa hanya nilai kosong dan duplikat membuat
dimensi ini bernilai 2 ke bawah.

---
> **Sebelum mengumpulkan:** jalankan **Kernel → Restart & Run All**. Notebook yang tidak dapat
> berjalan dari awal sampai akhir dianggap tidak memenuhi dimensi *ketepatan implementasi*.

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from pathlib import Path

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 60)

plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white'
})

DATA = '#0E7490'
SOROT = '#C2410C'
REDUP = '#5B6672'

AKAR = Path('..')
RAW = AKAR / 'data' / 'raw'
SIMULATED = AKAR / 'data' / 'simulated'
PROCESSED = AKAR / 'data' / 'processed'
FIGURES = AKAR / 'reports' / 'figures'

FIGURES.mkdir(parents=True, exist_ok=True)

## 0. Muat data mentah

Baca dari `data/raw`. Berkas ini **tidak pernah diubah** sepanjang proyek.

In [30]:
from google.colab import drive

drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
BERKAS = SIMULATED / '/content/drive/MyDrive/EDA sendiri/supermarket_sales_dirty.csv'

assert BERKAS.exists(), (
    f'Dataset tidak ditemukan: {BERKAS}\n'
    'Pastikan file supermarket_sales_dirty.csv berada di folder data/simulated.'
)

df = pd.read_csv(BERKAS)

print('Ukuran dataset:', df.shape)
print('Jumlah baris:', df.shape[0])
print('Jumlah kolom:', df.shape[1])

display(df.head())

Ukuran dataset: (1008, 17)
Jumlah baris: 1008
Jumlah kolom: 17


,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1008 entries, 0 to 1007
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1008 non-null   object 
 1   Branch                   1008 non-null   object 
 2   City                     1008 non-null   object 
 3   Customer type            1008 non-null   object 
 4   Gender                   1008 non-null   object 
 5   Product line             1008 non-null   object 
 6   Unit price               1008 non-null   float64
 7   Quantity                 1008 non-null   object 
 8   Tax 5%                   1008 non-null   float64
 9   Total                    1008 non-null   float64
 10  Date                     1008 non-null   object 
 11  Time                     1008 non-null   object 
 12  Payment                  1008 non-null   object 
 13  cogs                     1008 non-null   float64
 14  gross margin percentage 

## 1. Kelengkapan — nilai yang hilang

Jangan berhenti pada jumlahnya. Yang dinilai adalah apakah Anda memeriksa **polanya**:
apakah kekosongan berkaitan dengan kolom lain?

In [33]:
kosong = pd.DataFrame({
    'jumlah': df.isna().sum(),
    'persen': (df.isna().mean() * 100).round(2)
})

kosong = kosong[kosong['jumlah'] > 0].sort_values(
    'jumlah', ascending=False
)

if len(kosong):
    display(kosong)
else:
    print('Tidak ada nilai kosong.')

,jumlah,persen
Rating,18,1.79


In [34]:
KOLOM_X = 'Rating'
PEMBANDING = 'Total'

pembanding = df.groupby(
    df[KOLOM_X].isna()
)[PEMBANDING].agg(['count', 'mean', 'median'])

pembanding.index = ['Rating terisi', 'Rating kosong']

display(pembanding.round(2))

,count,mean,median
Rating terisi,990,323.45,255.80
Rating kosong,18,275.80,168.94


### Temuan Kelengkapan

Ditemukan nilai kosong pada kolom Rating. Nilai Rating yang kosong berjumlah 18 baris pada dataset simulasi.

Pemeriksaan dilakukan dengan membandingkan nilai Total antara transaksi yang memiliki Rating dan transaksi yang tidak memiliki Rating.

Perbedaan rata-rata Total dapat menjadi petunjuk adanya pola kekosongan. Namun, pemeriksaan ini belum cukup untuk memastikan mekanisme MCAR, MAR, atau MNAR.

## 2. Keunikan — baris kembar

In [35]:
# Memeriksa duplikasi seluruh baris
jumlah_duplikat = df.duplicated().sum()

print("Jumlah baris duplikat:", jumlah_duplikat)
print("Persentase duplikat:", round(
    jumlah_duplikat / len(df) * 100, 2
), "%")

# Memeriksa duplikasi Invoice ID
jumlah_invoice_duplikat = df['Invoice ID'].duplicated().sum()

print("\nJumlah Invoice ID duplikat:", jumlah_invoice_duplikat)

# Menampilkan baris yang terduplikasi
display(df[df.duplicated(keep=False)].sort_values('Invoice ID').head(20))

Jumlah baris duplikat: 8
Persentase duplikat: 0.79 %

Jumlah Invoice ID duplikat: 8


,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
536,339-12-4827,B,Mandalay,Member,Female,Fashion accessories,73.96,1,3.6980,77.6580,1/5/2019,11:32,Credit card,73.96,4.761905,3.6980,5.0
1007,339-12-4827,B,Mandalay,Member,Female,Fashion accessories,73.96,1,3.6980,77.6580,1/5/2019,11:32,Credit card,73.96,4.761905,3.6980,5.0
333,442-48-3607,A,Yangon,Member,Male,Food and beverages,23.48,2,2.3480,49.3080,3/14/2019,11:21,Credit card,46.96,4.761905,2.3480,7.9
1003,442-48-3607,A,Yangon,Member,Male,Food and beverages,23.48,2,2.3480,49.3080,3/14/2019,11:21,Credit card,46.96,4.761905,2.3480,7.9
604,484-22-8230,C,Naypyitaw,Member,Female,Fashion accessories,51.89,7,18.1615,381.3915,1/8/2019,20:08,Cash,363.23,4.761905,18.1615,4.5
1002,484-22-8230,C,Naypyitaw,Member,Female,Fashion accessories,51.89,7,18.1615,381.3915,1/8/2019,20:08,Cash,363.23,4.761905,18.1615,4.5
467,554-53-3790,B,Mandalay,Normal,Male,Sports and travel,37.02,6,11.1060,233.2260,3/22/2019,18:33,Cash,222.12,4.761905,11.1060,4.5
1006,554-53-3790,B,Mandalay,Normal,Male,Sports and travel,37.02,6,11.1060,233.2260,3/22/2019,18:33,Cash,222.12,4.761905,11.1060,4.5
735,607-65-2441,C,Naypyitaw,Member,Male,Health and beauty,81.95,10,40.9750,860.4750,3/10/2019,12:39,Credit card,819.50,4.761905,40.9750,6.0
1000,607-65-2441,C,Naypyitaw,Member,Male,Health and beauty,81.95,10,40.9750,860.4750,3/10/2019,12:39,Credit card,819.50,4.761905,40.9750,6.0


### Temuan Keunikan

Ditemukan 8 baris duplikat pada dataset simulasi. Duplikasi ini dapat menyebabkan transaksi dihitung lebih dari satu kali sehingga memengaruhi hasil analisis.

Baris duplikat perlu dihapus pada tahap cleaning. Pemeriksaan Invoice ID juga dilakukan untuk mengetahui apakah terdapat identitas transaksi yang berulang.

## 3. Konsistensi — penulisan kategori

Beda kapitalisasi dan spasi di ujung membuat satu kategori terpecah menjadi beberapa.

In [36]:
kategorikal = df.select_dtypes(include=['object', 'string']).columns.tolist()

for k in kategorikal:
    n_mentah  = df[k].nunique()
    n_rapi    = df[k].astype(str).str.strip().str.lower().nunique()
    tanda = '  <-- PERIKSA' if n_rapi < n_mentah else ''
    print(f'{k:24s} unik: {n_mentah:4d}   setelah dirapikan: {n_rapi:4d}{tanda}')

Invoice ID               unik: 1000   setelah dirapikan: 1000
Branch                   unik:    3   setelah dirapikan:    3
City                     unik:    6   setelah dirapikan:    3  <-- PERIKSA
Customer type            unik:    2   setelah dirapikan:    2
Gender                   unik:    2   setelah dirapikan:    2
Product line             unik:    6   setelah dirapikan:    6
Quantity                 unik:   11   setelah dirapikan:   11
Date                     unik:   89   setelah dirapikan:   89
Time                     unik:  506   setelah dirapikan:  506
Payment                  unik:    8   setelah dirapikan:    3  <-- PERIKSA


In [37]:
KOLOM = '____'          # TODO: kolom yang ditandai di atas
# print(sorted(df[KOLOM].dropna().unique()))

In [38]:
kolom_kategori = [
    'Branch',
    'City',
    'Customer type',
    'Gender',
    'Product line',
    'Payment'
]

for kolom in kolom_kategori:
    print(f"\n--- {kolom} ---")
    print(df[kolom].value_counts(dropna=False))


--- Branch ---
Branch
A    341
B    334
C    333
Name: count, dtype: int64

--- City ---
City
Yangon         339
Mandalay       330
Naypyitaw      329
Mandalay         4
Naypyitaw        4
Yangon           2
Name: count, dtype: int64

--- Customer type ---
Customer type
Member    506
Normal    502
Name: count, dtype: int64

--- Gender ---
Gender
Female    505
Male      503
Name: count, dtype: int64

--- Product line ---
Product line
Fashion accessories       181
Food and beverages        175
Electronic accessories    170
Sports and travel         169
Home and lifestyle        160
Health and beauty         153
Name: count, dtype: int64

--- Payment ---
Payment
Ewallet         345
Cash            343
Credit card     310
CASH              2
Credit card       2
CREDIT CARD       2
ewallet           2
cash              2
Name: count, dtype: int64


In [39]:
# Memeriksa spasi di awal atau akhir teks
for kolom in df.select_dtypes(include='object').columns:
    jumlah_spasi = df[kolom].astype(str).str.contains(
        r'^\s|\s$', regex=True
    ).sum()

    if jumlah_spasi > 0:
        print(f"{kolom}: {jumlah_spasi} nilai memiliki spasi berlebih")

City: 10 nilai memiliki spasi berlebih
Payment: 2 nilai memiliki spasi berlebih


### Temuan Konsistensi

Ditemukan ketidakkonsistenan pada beberapa nilai kategori, khususnya kolom Payment dan City.

Pada kolom Payment terdapat perbedaan format penulisan, seperti penggunaan huruf besar dan kecil. Pada kolom City terdapat spasi berlebih.

Ketidakkonsistenan tersebut perlu diperbaiki agar kategori yang sebenarnya sama tidak dianggap sebagai kategori berbeda ketika dilakukan analisis.

## 4. Validitas — nilai yang mustahil dan tipe data

Dua hal: apakah tipe datanya benar, dan apakah nilainya mungkin terjadi di dunia nyata.

In [40]:
print('TIPE DATA YANG PERLU DIPERIKSA')
for k in df.columns:
    if df[k].dtype == object:
        contoh = df[k].dropna().iloc[0] if df[k].notna().any() else ''
        print(f'  {k:24s} object   contoh: {contoh!r}')

TIPE DATA YANG PERLU DIPERIKSA
  Invoice ID               object   contoh: '750-67-8428'
  Branch                   object   contoh: 'A'
  City                     object   contoh: 'Yangon'
  Customer type            object   contoh: 'Member'
  Gender                   object   contoh: 'Female'
  Product line             object   contoh: 'Health and beauty'
  Quantity                 object   contoh: '7'
  Date                     object   contoh: '1/5/2019'
  Time                     object   contoh: '13:08'
  Payment                  object   contoh: 'Ewallet'


In [41]:
# Rentang nilai numerik -- cari nilai yang tidak mungkin (negatif, nol, terlalu besar)
num = df.select_dtypes('number')
print(num.describe().T[['min','25%','50%','75%','max']].round(2).to_string())

                           min     25%     50%     75%      max
Unit price               10.08   32.88   55.06   77.94    99.96
Tax 5%                    0.51    5.83   12.08   22.43    49.65
Total                    10.68  122.51  253.60  471.00  1042.65
cogs                     10.17  116.68  241.52  448.57   993.00
gross margin percentage   4.76    4.76    4.76    4.76     4.76
gross income              0.51    5.83   12.08   22.43    49.65
Rating                    0.00    5.50    6.95    8.40    11.00


In [42]:
# TODO: tulis pemeriksaan untuk setiap nilai mustahil yang Anda temukan
# contoh:
# print('durasi negatif  :', (df['durasi'] < 0).sum())
# print('umur > 120      :', (df['umur'] > 120).sum())

In [43]:
# Mengubah kolom numerik menjadi format angka sementara
rating_num = pd.to_numeric(df['Rating'], errors='coerce')
quantity_num = pd.to_numeric(df['Quantity'], errors='coerce')

# Rating seharusnya berada pada rentang 1 sampai 10
rating_invalid = df[
    rating_num.notna() &
    ~rating_num.between(1, 10)
]

# Quantity seharusnya berupa angka positif
quantity_invalid = df[
    quantity_num.isna() |
    (quantity_num <= 0)
]

print("Rating tidak valid:", len(rating_invalid))
print("Quantity tidak valid:", len(quantity_invalid))

print("\nContoh Rating tidak valid:")
display(rating_invalid[['Invoice ID', 'Rating']])

print("\nContoh Quantity tidak valid:")
display(quantity_invalid[['Invoice ID', 'Quantity']])

Rating tidak valid: 6
Quantity tidak valid: 4

Contoh Rating tidak valid:


,Invoice ID,Rating
62,347-34-2234,0.0
235,645-78-8093,11.0
268,144-51-6085,0.0
421,740-22-2500,0.0
657,562-12-5430,11.0
703,729-06-2010,11.0



Contoh Quantity tidak valid:


,Invoice ID,Quantity
40,280-35-5823,dua
54,242-55-6721,dua
392,563-36-9814,dua
579,207-73-1363,dua


> **Kolom tanggal:** bila format penulisannya hari/bulan/tahun, `pd.to_datetime` secara
> bawaan akan membacanya sebagai bulan/hari/tahun dan **tidak memunculkan pesan kesalahan**.
> Setiap tanggal yang harinya ≤ 12 akan tertukar secara diam-diam. Gunakan
> `format='%d/%m/%Y'`, lalu periksa `.min()`, `.max()`, dan jumlah bulan yang muncul.

### Temuan Validitas

Ditemukan nilai Rating yang berada di luar rentang 1 sampai 10, yaitu nilai 0 dan 11.

Selain itu, ditemukan nilai Quantity yang tidak dapat dibaca sebagai angka karena terdapat nilai teks "dua".

Nilai tersebut tidak sesuai dengan aturan tipe dan rentang data sehingga perlu diperbaiki pada tahap cleaning.

## 5. Akurasi — konsistensi antarkolom

Dimensi yang paling sering dilewatkan. Cari hubungan yang **harus** berlaku di data Anda,
lalu uji apakah benar-benar berlaku.

In [44]:
# contoh: total seharusnya = jumlah x harga_satuan
# seharusnya = df['jumlah'] * df['harga_satuan']
# selisih    = (df['total'] - seharusnya).abs()
# print('baris yang tidak cocok:', (selisih > 1).sum())

# TODO: tulis minimal satu pemeriksaan antarkolom untuk data Anda

In [45]:
# Mengubah kolom terkait menjadi numerik sementara
unit_price = pd.to_numeric(df['Unit price'], errors='coerce')
quantity = pd.to_numeric(df['Quantity'], errors='coerce')
cogs = pd.to_numeric(df['cogs'], errors='coerce')
tax = pd.to_numeric(df['Tax 5%'], errors='coerce')
total = pd.to_numeric(df['Total'], errors='coerce')

# Menghitung nilai berdasarkan rumus
cogs_hitung = unit_price * quantity
tax_hitung = cogs * 0.05
total_hitung = cogs + tax

# Membandingkan nilai aktual dan hasil perhitungan
toleransi = 0.01

hasil_akurasi = pd.DataFrame({
    'Pemeriksaan': [
        'COGS',
        'Tax 5%',
        'Total'
    ],
    'Jumlah tidak sesuai': [
        ((cogs - cogs_hitung).abs() > toleransi).sum(),
        ((tax - tax_hitung).abs() > toleransi).sum(),
        ((total - total_hitung).abs() > toleransi).sum()
    ]
})

display(hasil_akurasi)

,Pemeriksaan,Jumlah tidak sesuai
0,COGS,0
1,Tax 5%,0
2,Total,0


### Temuan Akurasi

Pemeriksaan akurasi dilakukan dengan membandingkan nilai COGS, Tax 5%, dan Total terhadap hasil perhitungan berdasarkan rumus transaksi.

Pemeriksaan ini bertujuan mengetahui apakah terdapat ketidaksesuaian perhitungan antar kolom. Baris dengan Quantity yang tidak valid belum dapat dinilai secara lengkap sebelum proses cleaning dilakukan.

## 6. Ringkasan temuan

Isi tabel ini. Tabel inilah yang disalin ke Data Quality Report dan menjadi dasar
urutan langkah di `02_cleaning.ipynb`.

In [46]:
ringkasan = pd.DataFrame({
    'Dimensi': [
        'Kelengkapan',
        'Keunikan',
        'Konsistensi',
        'Validitas',
        'Akurasi'
    ],
    'Pemeriksaan': [
        'Nilai kosong',
        'Baris duplikat',
        'Kategori dan spasi',
        'Rentang dan tipe data',
        'Hubungan perhitungan'
    ],
    'Hasil': [
        df.isna().sum().sum(),
        jumlah_duplikat,
        'Perlu normalisasi',
        len(rating_invalid) + len(quantity_invalid),
        'Lihat hasil pemeriksaan akurasi'
    ]
})

display(ringkasan)

,Dimensi,Pemeriksaan,Hasil
0,Kelengkapan,Nilai kosong,18
1,Keunikan,Baris duplikat,8
2,Konsistensi,Kategori dan spasi,Perlu normalisasi
3,Validitas,Rentang dan tipe data,10
4,Akurasi,Hubungan perhitungan,Lihat hasil pemeriksaan akurasi



## 6. Ringkasan temuan

| # | Masalah | Dimensi | Jumlah / bukti | Rencana penanganan |
|---|---------|---------|----------------|--------------------|
| 1 | Nilai kosong pada kolom Rating | Kelengkapan | 18 nilai kosong | Membiarkan nilai tetap kosong karena tidak ada dasar yang cukup untuk melakukan imputasi |
| 2 | Baris transaksi duplikat | Keunikan | 8 baris duplikat | Menghapus baris duplikat agar transaksi tidak dihitung lebih dari satu kali |
| 3 | Format kategori tidak konsisten dan spasi berlebih | Konsistensi | 12 nilai Payment tidak seragam dan 10 nilai City memiliki spasi berlebih | Menyeragamkan format kategori dan menghapus spasi berlebih |
| 4 | Nilai Rating dan Quantity tidak valid | Validitas | 6 Rating di luar rentang 1–10 dan 4 Quantity berupa teks "dua" | Mengubah Rating tidak valid menjadi nilai kosong dan mengonversi Quantity menjadi numerik |
| 5 | Hubungan perhitungan antarkolom perlu diperiksa | Akurasi | Pemeriksaan hubungan COGS, Tax 5%, dan Total | Mempertahankan nilai yang sesuai dan menyelidiki ketidaksesuaian jika ditemukan |



## Checklist sebelum mengumpulkan

- [x] Kelima dimensi diperiksa, bukan hanya nilai kosong dan duplikat.
- [x] Pola kekosongan diperiksa, bukan sekadar jumlahnya.
- [x] Minimal satu pemeriksaan konsistensi antarkolom ditulis.
- [x] Tabel ringkasan temuan terisi lengkap.
- [x] `data/raw` tidak berubah — notebook ini hanya membaca.
- [ ] Restart & Run All berjalan tanpa galat.
